In [28]:
import sqlite3

import pandas as pd

conn = sqlite3.connect("../incentive-util.db")


def get_item_id_set():
    df = pd.read_sql_query("SELECT id FROM items", conn)
    return set(df["id"])


def insert(to_insert: list):
    conn.executemany(
        """INSERT INTO items (id, name, type, brand, unit, base_price, sell_price)
        VALUES (:id, :name, :type, :brand, :unit, :base_price, :sell_price)""",
        to_insert,
    )
    conn.commit()


def update(to_update: list):
    conn.executemany(
        """UPDATE items
        SET name=:name, type=:type, brand=:brand, unit=:unit,
            base_price=:base_price, sell_price=:sell_price
        WHERE id=:id""",
        to_update,
    )
    conn.commit()

In [27]:
def build_cleaned_list(import_path: str):
    df = pd.read_excel(import_path)
    df_clean = df.dropna(how="all").dropna(how="all", axis=1)
    ITEM_ID = 0
    ITEM_NAME = 1
    ITEM_TYPE = 2
    ITEM_BRAND = 3
    ITEM_UNIT = 5
    ITEM_BASE_PRICE = 6
    ITEM_SELL_PRICE = 7
    return [
        {
            "id": row[ITEM_ID],
            "name": row[ITEM_NAME],
            "type": row[ITEM_TYPE],
            "brand": row[ITEM_BRAND],
            "unit": row[ITEM_UNIT],
            "base_price": row[ITEM_BASE_PRICE],
            "sell_price": row[ITEM_SELL_PRICE],
        }
        for row in df_clean.itertuples(index=False)
    ]


def filter_to_insert(existing_ids: set, items: list):
    return [i for i in items if i["id"] not in existing_ids]

def filter_to_update(existing_ids: set, items: list):
    return [i for i in items if i["id"] in existing_ids]

In [ ]:
import_path = "items.xlsx"
items = build_cleaned_list(import_path)
existing_ids = get_item_id_set()
to_insert = filter_to_insert(existing_ids, items)
to_update = filter_to_update(existing_ids, items)

insert(to_insert)
update(to_update)


/home/pepis/Work/incentive-util/.venv/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
